# Inspect the dataset manifest
Quick sanity-checking and stats on the treatment/control manifest before moving into the actual ML (CAE) phase.

## Setup

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd() / "scripts"))

import pandas as pd
from scripts.dataset_builder import build_dataset_manifest

pd.set_option("display.max_columns", None)


ModuleNotFoundError: No module named 'pandas_market_calendars'

## Build the manifest

In [ ]:
manifest = build_dataset_manifest()
manifest.shape


## Overview

In [ ]:
manifest.head()


In [ ]:
manifest.dtypes


In [ ]:
manifest["label"].value_counts()


## Per-ticker breakdown

In [ ]:
manifest.groupby(["ticker", "label"]).size().unstack(fill_value=0)


## Treatment events: AMC vs BMO split

In [ ]:
treatment = manifest[manifest["label"] == "treatment"]
treatment["time_of_day"].value_counts()


In [ ]:
treatment.groupby(["ticker", "time_of_day"]).size().unstack(fill_value=0)


## Date range coverage

In [ ]:
print("earliest t_minus_1:", manifest["t_minus_1"].min())
print("latest t0:", manifest["t0"].max())


## Sanity checks

In [ ]:
# no row should be missing ticker/t_minus_1/t0/label
manifest[["ticker", "t_minus_1", "t0", "label"]].isnull().sum()


In [ ]:
# treatment rows should have full metadata, control rows should have none
treatment_nulls = manifest[manifest["label"] == "treatment"][["fiscal_period", "earnings_date", "time_of_day"]].isnull().sum()
control_nulls = manifest[manifest["label"] == "control"][["fiscal_period", "earnings_date", "time_of_day"]].isnull().sum()
print("treatment nulls (want all 0):\n", treatment_nulls)
print("\ncontrol nulls (want == total control row count):\n", control_nulls)


In [ ]:
# no (ticker, t_minus_1) pair should appear under both labels
overlap = manifest.groupby(["ticker", "t_minus_1"])["label"].nunique()
overlap[overlap > 1]


## Save a snapshot for reuse (optional)

In [ ]:
# manifest.to_parquet("data/processed/manifest.parquet")
